# Solution R6 · Warm-up: the first PCA, explained variance and loadings

**Type**: Implement with scikit-learn

## Objective

Apply PCA to 13 measurements of wines: how much variance each component explains, how many are needed for 90%, which variables weigh on each one, and what happens if you forget to standardize.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_wine(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## Commented solution

In [ ]:
Z = StandardScaler().fit_transform(X)
pca = PCA().fit(Z)
ev = pca.explained_variance_ratio_
ev2 = float(ev[:2].sum())
n90 = int(np.searchsorted(np.cumsum(ev), 0.9)) + 1
print("explained variance (first 4):", ev[:4].round(3), "| first two:", round(ev2, 3), "| components for 90%:", n90)

In [ ]:
loadings = pd.DataFrame(pca.components_[:2].T, index=X.columns, columns=["PC1", "PC2"])
pc1_variable, pc2_variable = loadings["PC1"].abs().idxmax(), loadings["PC2"].abs().idxmax()
print(loadings.round(2).reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(4).to_string())
print("PC1:", pc1_variable, "| PC2:", pc2_variable)

In [ ]:
unscaled_ev1 = float(PCA().fit(X).explained_variance_ratio_[0])
print(f"first component without standardizing: {unscaled_ev1:.4f} of the variance")

In [ ]:
acc_2comp = make_pipeline(StandardScaler(), PCA(2), LogisticRegression()).fit(X_tr, y_tr).score(X_te, y_te)
acc_13 = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr).score(X_te, y_te)
print(f"test accuracy: 2 components {acc_2comp:.3f} | 13 variables {acc_13:.3f}")

## Check

In [ ]:
with verify():
    check(close(ev.sum(), 1.0) and 0.5 < ev2 < 0.6 and n90 == 8, "the components sum to 1; the first two explain 55%; 8 are needed for 90%", "np.cumsum(ev), np.searchsorted(...)")
    check(pc1_variable == "flavanoids" and pc2_variable == "color_intensity", "PC1 is dominated by flavanoids, PC2 by color_intensity", "loadings['PC1'].abs().idxmax()")
    check(unscaled_ev1 > 0.99, "without standardizing the first component explains over 99%: it is just `proline`", "PCA().fit(X).explained_variance_ratio_[0]")
    check(0.85 < acc_2comp < acc_13 <= 1.0, "with 2 components the accuracy is high (over 0.85) but lower than with all the 13 variables", "make_pipeline(StandardScaler(), LogisticRegression())")
done()

## What you learned

- **The explained variance is read cumulatively:** the first component is worth 36%, the first two 55%, and 8 out of 13 are needed to reach 90%. Here the data do not compress much: the variables are informative and only partly redundant.
- **The loadings say what a component represents:** PC1 is dominated by `flavanoids` (with `total_phenols` and `od280/od315`: the measurements linked to phenols), PC2 by `color_intensity`. A loading that is large in absolute value means that variable weighs a lot on the component.
- **Without standardizing PCA is useless:** the first component explains 99.8% of the variance, it is simply `proline` (standard deviation 315 against 0.1-14 for the others). PCA maximizes the variance in absolute value, so it follows whoever has the largest scale.
- **Fewer components, less information:** with 2 components the logistic regression reaches 0.93 accuracy on the test set, with all the 13 variables 1.0. The compression has a cost, which you measure (lesson 6.3).

## Common pitfall

Doing PCA on variables with different scales without standardizing: the first component is the variable with the largest scale and the result looks "very informative" (99% of the variance) without being so.

## Interview question

<details><summary>What does it mean that the first two components explain 55% of the variance?</summary>

That projecting the data onto the plane of the first two components preserves 55% of the total variance (of the standardized variables): it is a measure of reconstruction fidelity, not of usefulness for predicting something. It is looked at together with the ability of the model to use the components.

</details>